In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/store .csv')
df

,ปี,จังหวัด,อำเภอ,อปท,คร.ที่กำจัดขยะ
0,2563,นนทบุรี,ปากเกร็ด,เทศบาลนครปากเกร็ด,"189,356"
1,2563,นนทบุรี,บางกรวย,เทศบาลเมืองบางกรวย,"22,106"
2,2563,นนทบุรี,เมืองนนทบุรี,เทศบาลเมืองบางกร่าง,"7,332"
3,2563,นนทบุรี,บางบัวทอง,เทศบาลเมืองบางบัวทอง,"25,269"
4,2563,นนทบุรี,บางใหญ่,เทศบาลเมืองบางแม่นาง,"23,655"
...,...,...,...,...,...
147,2567,นนทบุรี,ไทรน้อย,อบต.ราษฎร์นิยม,"1,157"
148,2567,นนทบุรี,บางบัวทอง,อบต.ละหาร,"13,162"
149,2567,นนทบุรี,บางบัวทอง,อบต.ลำโพ,"8,611"
150,2567,นนทบุรี,ไทรน้อย,อบต.หนองเพรางาย,NaN


In [3]:
df.isnull().sum()

ปี                 0
จังหวัด            0
อำเภอ              0
อปท                0
คร.ที่กำจัดขยะ    46
dtype: int64

## การจัดการค่าสูญหาย (Missing Value Treatment)

**ปัญหาที่พบ:**
1. `คร.ที่กำจัดขยะ` เก็บเป็น string (มีเครื่องหมาย `,`) ต้องแปลงเป็น numeric ก่อน
2. มีค่าสูญหาย 46 แถว (~30%)

**กลยุทธ์การเติมค่า (Imputation Strategy):**
เนื่องจากข้อมูลเป็น Panel Data (อปท. เดิมวัดซ้ำหลายปี) จึงใช้ลำดับดังนี้
1. เติมด้วย **ค่ามัธยฐานของ อปท. เดียวกัน** ข้ามปี (อปท. ขนาดต่างกันมาก)
2. หากยังขาด → เติมด้วย **ค่ามัธยฐานของ อำเภอ เดียวกัน** (ใกล้เคียงตามภูมิศาสตร์)
3. หากยังขาด → เติมด้วย **ค่ามัธยฐานรวม** (Fallback)

In [4]:
# ── Step 1: วิเคราะห์ค่าสูญหาย ──────────────────────────────────────────
print("=== ค่าสูญหายแยกตามปี ===")
print(df.groupby('ปี')['คร.ที่กำจัดขยะ'].apply(lambda x: x.isnull().sum()).rename('จำนวนที่หาย'))
print()
print("=== ค่าสูญหายแยกตามอำเภอ ===")
print(df.groupby('อำเภอ')['คร.ที่กำจัดขยะ'].apply(lambda x: x.isnull().sum()).rename('จำนวนที่หาย'))
print()
print("=== อปท. ที่มีค่าสูญหายทุกปี (ไม่มีข้อมูลเลย) ===")
all_null_apt = (df.groupby('อปท')['คร.ที่กำจัดขยะ']
                  .apply(lambda x: x.isnull().all()))
print(all_null_apt[all_null_apt].index.tolist())

=== ค่าสูญหายแยกตามปี ===
ปี
2563     0
2564     0
2565     0
2566    25
2567    21
Name: จำนวนที่หาย, dtype: int64

=== ค่าสูญหายแยกตามอำเภอ ===
อำเภอ
บางกรวย          5
บางบัวทอง        9
บางใหญ่          7
ปากเกร็ด         9
เมืองนนทบุรี     4
ไทรน้อย         12
Name: จำนวนที่หาย, dtype: int64

=== อปท. ที่มีค่าสูญหายทุกปี (ไม่มีข้อมูลเลย) ===
['อบต.บางขุนกอง', 'อบต.หนองเพรางาย', 'เทศบาลนครนนทบุรี', 'เทศบาลนครบางบัวทอง', 'เทศบาลเมืองพิมลราช']


In [5]:
# ── Step 2: แปลงชนิดข้อมูลและเติมค่าสูญหาย ───────────────────────────────
df_clean = df.copy()

# 2a. แปลง string → numeric (ลบเครื่องหมาย ,)
df_clean['คร.ที่กำจัดขยะ'] = (
    df_clean['คร.ที่กำจัดขยะ']
    .astype(str)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
    .pipe(pd.to_numeric, errors='coerce')
)

before = df_clean['คร.ที่กำจัดขยะ'].isnull().sum()
print(f"NaN ก่อนเติม: {before}")

# 2b. เติมด้วยค่ามัธยฐานของ อปท. เดียวกัน (วิธีที่ 1)
df_clean['คร.ที่กำจัดขยะ'] = (
    df_clean.groupby('อปท')['คร.ที่กำจัดขยะ']
    .transform(lambda x: x.fillna(x.median()))
)

after_step1 = df_clean['คร.ที่กำจัดขยะ'].isnull().sum()
print(f"NaN หลังเติมด้วย median ของ อปท.: {after_step1}")

# 2c. เติมด้วยค่ามัธยฐานของ อำเภอ เดียวกัน (วิธีที่ 2 — fallback)
df_clean['คร.ที่กำจัดขยะ'] = (
    df_clean.groupby('อำเภอ')['คร.ที่กำจัดขยะ']
    .transform(lambda x: x.fillna(x.median()))
)

after_step2 = df_clean['คร.ที่กำจัดขยะ'].isnull().sum()
print(f"NaN หลังเติมด้วย median ของ อำเภอ: {after_step2}")

# 2d. เติมด้วย median รวม (วิธีที่ 3 — last resort)
global_median = df_clean['คร.ที่กำจัดขยะ'].median()
df_clean['คร.ที่กำจัดขยะ'] = df_clean['คร.ที่กำจัดขยะ'].fillna(global_median)

after_step3 = df_clean['คร.ที่กำจัดขยะ'].isnull().sum()
print(f"NaN หลังเติมด้วย median รวม (fallback): {after_step3}")

# แปลงเป็น int (ครัวเรือนไม่มีทศนิยม)
df_clean['คร.ที่กำจัดขยะ'] = df_clean['คร.ที่กำจัดขยะ'].round().astype(int)

print(f"\nสรุป: เติมค่าทั้งหมด {before} แถว เสร็จสมบูรณ์")
df_clean

NaN ก่อนเติม: 46
NaN หลังเติมด้วย median ของ อปท.: 10
NaN หลังเติมด้วย median ของ อำเภอ: 0
NaN หลังเติมด้วย median รวม (fallback): 0

สรุป: เติมค่าทั้งหมด 46 แถว เสร็จสมบูรณ์


,ปี,จังหวัด,อำเภอ,อปท,คร.ที่กำจัดขยะ
0,2563,นนทบุรี,ปากเกร็ด,เทศบาลนครปากเกร็ด,189356
1,2563,นนทบุรี,บางกรวย,เทศบาลเมืองบางกรวย,22106
2,2563,นนทบุรี,เมืองนนทบุรี,เทศบาลเมืองบางกร่าง,7332
3,2563,นนทบุรี,บางบัวทอง,เทศบาลเมืองบางบัวทอง,25269
4,2563,นนทบุรี,บางใหญ่,เทศบาลเมืองบางแม่นาง,23655
...,...,...,...,...,...
147,2567,นนทบุรี,ไทรน้อย,อบต.ราษฎร์นิยม,1157
148,2567,นนทบุรี,บางบัวทอง,อบต.ละหาร,13162
149,2567,นนทบุรี,บางบัวทอง,อบต.ลำโพ,8611
150,2567,นนทบุรี,ไทรน้อย,อบต.หนองเพรางาย,1622


In [6]:
# ── Step 3: ตรวจสอบและบันทึก ─────────────────────────────────────────────
print("=== ตรวจสอบหลังทำความสะอาด ===")
df_clean.info()
print()
print(f"ค่าสูญหายคงเหลือ: {df_clean.isnull().sum().sum()}")
print()
print("=== สถิติเบื้องต้นของ คร.ที่กำจัดขยะ ===")
print(df_clean['คร.ที่กำจัดขยะ'].describe().apply(lambda x: f"{x:,.0f}"))

=== ตรวจสอบหลังทำความสะอาด ===
<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   ปี              152 non-null    int64
 1   จังหวัด         152 non-null    str  
 2   อำเภอ           152 non-null    str  
 3   อปท             152 non-null    str  
 4   คร.ที่กำจัดขยะ  152 non-null    int64
dtypes: int64(2), str(3)
memory usage: 6.1 KB

ค่าสูญหายคงเหลือ: 0

=== สถิติเบื้องต้นของ คร.ที่กำจัดขยะ ===
count        152
mean      11,740
std       20,300
min            0
25%        1,891
50%        5,484
75%       12,672
max      189,356
Name: คร.ที่กำจัดขยะ, dtype: str


In [7]:
for col in df_clean.columns[1:4]:
    df_clean[col] = df_clean[col].astype('category')
df_clean.info()
df_clean.to_csv('../clean_data/clean_store.csv', index=False)
print("\nบันทึก clean data สำเร็จ: ../clean_data/clean_store.csv")

<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   ปี              152 non-null    int64   
 1   จังหวัด         152 non-null    category
 2   อำเภอ           152 non-null    category
 3   อปท             152 non-null    category
 4   คร.ที่กำจัดขยะ  152 non-null    int64   
dtypes: category(3), int64(2)
memory usage: 3.4 KB

บันทึก clean data สำเร็จ: ../clean_data/clean_store.csv
